In [8]:
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, TargetEncoder,
    StandardScaler, MinMaxScaler, RobustScaler,
    PolynomialFeatures
)

from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, QuantileRegressor
)

from sklearn.metrics import (
    r2_score,
    root_mean_squared_error as rmse,
    mean_absolute_percentage_error as mape,
)

from sklearn.compose import (
    ColumnTransformer
)

from sklearn.impute import (
    SimpleImputer
)

import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsRegressor

from sklearn.svm import (
    LinearSVR
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.kernel_ridge import KernelRidge

In [2]:
df = pd.read_csv("../../data/data_transformed2_misha.csv").set_index(["ID на сайте", "Источник"])
df.head()

,,Цена,Дата,Тип автора,Город,Метро/Район,Адрес,Описание,lat,lng,URL,...,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9
ID на сайте,Источник,,,,,,,,,,,,,,,,,,,,,
321970859,cian.ru,34500.0,2025-12-17 08:36:40,Агентство,Химки,Беломорская,"Рабочая ул., 2",название офис московский область химки рабочи...,55.908036,37.432951,https://www.cian.ru/rent/commercial/321970859,...,0.001053,0.001053,0.106424,0.001053,0.001053,0.001053,0.487249,0.001053,0.212217,0.187793
323570959,cian.ru,160500.0,2025-12-17 08:36:40,Частное лицо,Солнечногорск городской округ,Зеленоград — Крюково,21А,название торговый площадь московский область ...,55.972650,37.099604,https://www.cian.ru/rent/commercial/323570959,...,0.757324,0.001205,0.001205,0.201295,0.001205,0.032945,0.001205,0.001205,0.001205,0.001205
319937619,cian.ru,1226740.0,2025-12-17 08:36:40,Агентство,Мытищи,Медведково,"ул. Колпакова, 46А",название офис московский область мытищи ул ко...,55.925178,37.718300,https://www.cian.ru/rent/commercial/319937619,...,0.000953,0.068630,0.000953,0.000953,0.169779,0.000952,0.607339,0.000953,0.148537,0.000953
319936797,cian.ru,236480.0,2025-12-17 08:36:39,Агентство,Мытищи,Медведково,"ул. Колпакова, 46А",название офис московский область мытищи ул ко...,55.925178,37.718300,https://www.cian.ru/rent/commercial/319936797,...,0.000953,0.068630,0.000953,0.000953,0.169779,0.000952,0.607339,0.000953,0.148537,0.000953
319938233,cian.ru,1537120.0,2025-12-17 08:36:38,Агентство,Мытищи,Медведково,"ул. Колпакова, 46А",название офис московский область мытищи ул ко...,55.925178,37.718300,https://www.cian.ru/rent/commercial/319938233,...,0.000953,0.068630,0.000953,0.000953,0.169779,0.000952,0.607339,0.000953,0.148537,0.000953


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 29559 entries, (np.int64(321970859), 'cian.ru') to (np.int64(4937488939), 'avito.ru')
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Цена                     29559 non-null  float64
 1   Дата                     29559 non-null  object 
 2   Тип автора               29559 non-null  object 
 3   Город                    29559 non-null  object 
 4   Метро/Район              22098 non-null  object 
 5   Адрес                    29232 non-null  object 
 6   Описание                 29559 non-null  object 
 7   lat                      29559 non-null  float64
 8   lng                      29559 non-null  float64
 9   URL                      29559 non-null  object 
 10  Ссылки на картинки       29108 non-null  object 
 11  Расстояние до метро, км  20605 non-null  float64
 12  Этаж                     28639 non-null  float64
 13  Этажность здания

In [4]:
n_components = 10
claster_column_names = ["Cluster " + str(i) for i in range(n_components)]
nonclaster_column_names = ["Общая площадь", "Расстояние до метро, км", "Вид объекта", "Тип автора", "Москва?", "Этаж", "Этажность здания"]
column_names = claster_column_names + nonclaster_column_names

In [5]:
column_names

['Cluster 0',
 'Cluster 1',
 'Cluster 2',
 'Cluster 3',
 'Cluster 4',
 'Cluster 5',
 'Cluster 6',
 'Cluster 7',
 'Cluster 8',
 'Cluster 9',
 'Общая площадь',
 'Расстояние до метро, км',
 'Вид объекта',
 'Тип автора',
 'Москва?',
 'Этаж',
 'Этажность здания']

In [6]:
y = df["Цена"].copy()
X = df[column_names].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=X["Вид объекта"])

In [7]:
ct1 = ColumnTransformer(
    transformers=[
        ("isna", SimpleImputer(strategy="constant", fill_value=1, add_indicator=True, copy=False), ["Расстояние до метро, км", "Этаж", "Этажность здания"]),
    ],
    remainder="passthrough",
    n_jobs=-1,
    verbose_feature_names_out=False
).set_output(transform="pandas")
ct1.fit(X_train)

ct2 = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), ["Общая площадь", "Расстояние до метро, км"]),
        ("onehot", OneHotEncoder(sparse_output=False), ["Вид объекта"]),
        ("ordinal", OrdinalEncoder(), ["Тип автора"]),
    ],
    remainder="passthrough",
    n_jobs=-1,
    verbose_feature_names_out=False
).set_output(transform="pandas")
ct2.fit(ct1.transform(X_train))

ct3 = ColumnTransformer(
    transformers=[
        ("polynoms", PolynomialFeatures(degree=3), ["Общая площадь", "Расстояние до метро, км", "Этаж"]),
        # ("mult", PolynomialFeatures(degree=3), [["Общая площадь", "Этаж"]])
    ],
    remainder="passthrough",
    n_jobs=-1,
).set_output(transform="pandas")
ct3.fit(ct2.transform(ct1.transform(X_train)))

def transform_all(X):
    return ct3.transform(ct2.transform(ct1.transform(X)))

In [9]:
X_train_pre = transform_all(X_train)
X_test_pre = transform_all(X_test)

In [22]:
model = DecisionTreeRegressor(
    max_depth=8
).fit(X_train_pre, y_train)

y_train_pred = model.predict(X_train_pre)
y_test_pred = model.predict(X_test_pre)

In [23]:
r2_score(y_train, y_train_pred), rmse(y_train, y_train_pred), mape(y_train, y_train_pred)

(0.8497020531250854, 163877.95216418532, 0.44925562218948906)

In [24]:
r2_score(y_test, y_test_pred), rmse(y_test, y_test_pred), mape(y_test, y_test_pred)

(0.7705917769691368, 201986.59354277508, 0.48243326739303327)